# Phase 3a: Synthetic Tamil Instruction Data — Local via Ollama

**Goal**: Generate 50K Tamil instruction pairs with zero API cost using a local Ollama model.

**Requirements**:
- [Ollama](https://ollama.ai) installed locally
- Pull the generation model: `ollama pull llama3.1:8b-instruct-q4_K_M`
- Fits on **8GB VRAM** (4-bit quantized)

**Generates 4 instruction types**:
1. Q&A extraction from Wikipedia chunks
2. Summarization (compress to 2–3 sentences)
3. Fill-in-the-blank (mask named entities)
4. Instruction paraphrase ("Describe X")

**Output**: JSONL file + HuggingFace dataset

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["ollama", "datasets", "tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("Ready.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
import os

# Ollama model to use for generation
# llama3.1:8b-instruct-q4_K_M fits 8GB VRAM and speaks Tamil
OLLAMA_MODEL     = "llama3.1:8b-instruct-q4_K_M"

# Source data
LOCAL_DATA_FILE  = "../data/tawiki_pages.jsonl"   # full articles (richer context)

# How many instruction pairs to generate per type
N_QA             = 15000
N_SUMMARY        = 10000
N_FILL_BLANK     = 10000
N_DESCRIBE       = 15000

# Output
OUTPUT_FILE      = "../data/synthetic_instructions.jsonl"
HF_OUTPUT_REPO   = "wickkiey/tamil-synthetic-instructions"
HF_TOKEN         = None

# Quality filters for generated text
MIN_RESPONSE_LEN = 20     # minimum chars in generated response
MIN_TAMIL_RATIO  = 0.40   # lower threshold — model may mix some English

print(f"Generator model : {OLLAMA_MODEL}")
print(f"Target total    : {N_QA + N_SUMMARY + N_FILL_BLANK + N_DESCRIBE:,} pairs")

In [ ]:
# ── Verify Ollama is running ───────────────────────────────────────────────────
import ollama

try:
    models = ollama.list()
    model_names = [m["name"] for m in models.get("models", [])]
    print(f"Ollama running. Available models: {model_names}")
    if OLLAMA_MODEL not in model_names and not any(OLLAMA_MODEL in n for n in model_names):
        print(f"\n⚠ Model '{OLLAMA_MODEL}' not found. Pull it first:")
        print(f"  ollama pull {OLLAMA_MODEL}")
except Exception as e:
    print(f"❌ Ollama not running: {e}")
    print("Start Ollama: run 'ollama serve' in a terminal, then re-run this cell.")

In [ ]:
# ── Load Wikipedia articles ────────────────────────────────────────────────────
import json
from pathlib import Path

articles = []
with open(LOCAL_DATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        try:
            obj = json.loads(line)
            title = obj.get("title", "")
            text  = obj.get("wikitext", obj.get("text", ""))
            # Take first 800 chars as context (enough for Q&A, not too long for 8GB)
            if title and text and len(text.strip()) > 100:
                articles.append({"title": title, "text": text.strip()[:800]})
        except json.JSONDecodeError:
            continue

import random
random.seed(42)
random.shuffle(articles)

print(f"Loaded {len(articles):,} articles")
print(f"Sample: {articles[0]['title']} — {articles[0]['text'][:100]}...")

In [ ]:
# ── Helper: quality check on generated text ────────────────────────────────────
TAMIL_START = 0x0B80
TAMIL_END   = 0x0BFF

def tamil_ratio(text: str) -> float:
    if not text:
        return 0.0
    return sum(1 for c in text if TAMIL_START <= ord(c) <= TAMIL_END) / len(text)

def is_valid_response(text: str) -> bool:
    if not text or len(text.strip()) < MIN_RESPONSE_LEN:
        return False
    if tamil_ratio(text) < MIN_TAMIL_RATIO:
        return False
    return True

In [ ]:
# ── Helper: call Ollama ────────────────────────────────────────────────────────
def generate(system_prompt: str, user_prompt: str, temperature: float = 0.7) -> str:
    """Call local Ollama model and return response text."""
    try:
        response = ollama.chat(
            model   = OLLAMA_MODEL,
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            options = {"temperature": temperature, "num_predict": 300},
        )
        return response["message"]["content"].strip()
    except Exception as e:
        return ""

In [ ]:
# ── Generation Type 1: Q&A ─────────────────────────────────────────────────────
from tqdm import tqdm

SYSTEM_QA = (
    "நீங்கள் ஒரு தமிழ் மொழி வல்லுநர். "
    "கொடுக்கப்பட்ட உரையிலிருந்து ஒரு கேள்வி மற்றும் தெளிவான பதில் தமிழில் எழுதுங்கள். "
    "வெளியீடு வடிவம்: கேள்வி: <question>\nபதில்: <answer>"
)

qa_pairs = []
article_pool = articles * (N_QA // len(articles) + 1)  # cycle if needed

for i, article in enumerate(tqdm(article_pool[:N_QA * 2], desc="Q&A generation")):
    if len(qa_pairs) >= N_QA:
        break
    prompt = f"உரை:\n{article['text']}\n\nகேள்வி மற்றும் பதில் எழுதுங்கள்:"
    response = generate(SYSTEM_QA, prompt)
    if not is_valid_response(response):
        continue
    # Parse Q/A split
    lines = response.split("\n")
    question = next((l.replace("கேள்வி:", "").strip() for l in lines if "கேள்வி:" in l), "")
    answer   = next((l.replace("பதில்:", "").strip()  for l in lines if "பதில்:"  in l), response)
    if question and answer:
        qa_pairs.append({
            "instruction": question,
            "input":       "",
            "output":      answer,
            "type":        "qa",
            "source":      article["title"]
        })

print(f"Q&A pairs generated: {len(qa_pairs):,}")

In [ ]:
# ── Generation Type 2: Summarization ──────────────────────────────────────────
SYSTEM_SUMMARY = (
    "நீங்கள் ஒரு தமிழ் மொழி வல்லுநர். "
    "கொடுக்கப்பட்ட உரையை 2-3 வாக்கியங்களில் சுருக்கமாக தமிழில் எழுதுங்கள்."
)

summary_pairs = []
for article in tqdm(article_pool[:N_SUMMARY * 2], desc="Summarization"):
    if len(summary_pairs) >= N_SUMMARY:
        break
    prompt   = f"இந்த உரையை சுருக்கி எழுதுங்கள்:\n{article['text']}"
    response = generate(SYSTEM_SUMMARY, prompt, temperature=0.5)
    if not is_valid_response(response):
        continue
    summary_pairs.append({
        "instruction": f"{article['title']} பற்றி சுருக்கமாக விளக்குங்கள்.",
        "input":       "",
        "output":      response,
        "type":        "summary",
        "source":      article["title"]
    })

print(f"Summary pairs generated: {len(summary_pairs):,}")

In [ ]:
# ── Generation Type 3: Fill-in-the-blank ──────────────────────────────────────
import re

SYSTEM_FILL = (
    "நீங்கள் ஒரு தமிழ் மொழி ஆசிரியர். "
    "வழங்கப்பட்ட வாக்கியத்தில் காலி இடத்தை நிரப்புங்கள். "
    "பதில் மட்டும் தமிழில் தருங்கள்."
)

fill_pairs = []
for article in tqdm(article_pool[:N_FILL_BLANK * 2], desc="Fill-in-blank"):
    if len(fill_pairs) >= N_FILL_BLANK:
        break
    # Extract first sentence and mask the first noun-like word (capitalized or after space)
    first_sent = article["text"].split("\n")[0][:200]
    words = first_sent.split()
    if len(words) < 5:
        continue
    # Mask word at position 2–4 (usually a key noun)
    mask_idx = min(3, len(words) - 1)
    answer_word = words[mask_idx]
    masked = words.copy()
    masked[mask_idx] = "______"
    masked_sent = " ".join(masked)

    if len(answer_word) < 2:
        continue

    fill_pairs.append({
        "instruction": f"காலி இடத்தை நிரப்புங்கள்: {masked_sent}",
        "input":       "",
        "output":      answer_word,
        "type":        "fill_blank",
        "source":      article["title"]
    })

print(f"Fill-in-blank pairs: {len(fill_pairs):,}")

In [ ]:
# ── Generation Type 4: Describe / Explain ─────────────────────────────────────
SYSTEM_DESCRIBE = (
    "நீங்கள் ஒரு தமிழ் மொழி வல்லுநர். "
    "கேட்கப்படும் தலைப்பை தமிழில் தெளிவாக விளக்குங்கள்."
)

desc_pairs = []
templates = [
    "{title} என்றால் என்ன?",
    "{title} பற்றி விளக்குங்கள்.",
    "{title} பற்றி சுருக்கமாக கூறுங்கள்.",
    "{title} எவ்வாறு இயங்குகிறது?",
    "{title} ஏன் முக்கியமானது?",
]

for i, article in enumerate(tqdm(article_pool[:N_DESCRIBE * 2], desc="Describe/Explain")):
    if len(desc_pairs) >= N_DESCRIBE:
        break
    instruction = templates[i % len(templates)].format(title=article["title"])
    context_prompt = f"கீழ்க்கண்ட தகவல்களை பயன்படுத்தி பதில் தருங்கள்:\n{article['text'][:400]}\n\n{instruction}"
    response = generate(SYSTEM_DESCRIBE, context_prompt, temperature=0.6)
    if not is_valid_response(response):
        continue
    desc_pairs.append({
        "instruction": instruction,
        "input":       "",
        "output":      response,
        "type":        "describe",
        "source":      article["title"]
    })

print(f"Describe pairs generated: {len(desc_pairs):,}")

In [ ]:
# ── Combine & save ─────────────────────────────────────────────────────────────
all_pairs = qa_pairs + summary_pairs + fill_pairs + desc_pairs
random.shuffle(all_pairs)

# Save as JSONL
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for pair in all_pairs:
        f.write(json.dumps(pair, ensure_ascii=False) + "\n")

print(f"Total pairs     : {len(all_pairs):,}")
print(f"  Q&A           : {len(qa_pairs):,}")
print(f"  Summarization : {len(summary_pairs):,}")
print(f"  Fill-in-blank : {len(fill_pairs):,}")
print(f"  Describe      : {len(desc_pairs):,}")
print(f"Saved to        : {OUTPUT_FILE}")

In [ ]:
# ── Push to HuggingFace ────────────────────────────────────────────────────────
from datasets import Dataset

ds = Dataset.from_list(all_pairs)
print(f"Dataset: {ds}")
print(f"\nSample:\n{ds[0]}")

PUSH_TO_HF = True
if PUSH_TO_HF:
    ds.push_to_hub(
        HF_OUTPUT_REPO,
        token          = HF_TOKEN,
        commit_message = "Tamil synthetic instructions — 50K pairs from Wikipedia (local Ollama generation)"
    )
    print(f"Pushed to: https://huggingface.co/datasets/{HF_OUTPUT_REPO}")

## Tips for Overnight Local Generation

1. Run in VS Code terminal: `jupyter nbconvert --to notebook --execute evaluation/01_generate_synthetic_local.ipynb`
2. Ollama handles context and memory automatically — you can close the browser
3. Output is written incrementally to `data/synthetic_instructions.jsonl` — safe to interrupt
4. For faster generation on 8GB: switch to `OLLAMA_MODEL = "qwen2.5:3b-instruct-q4_K_M"` (smaller, faster)

## Next Step
Once you have 50K pairs, proceed to `finetuning/02_sft_instruction_tuning.ipynb`.